PRESS describes the decisions and actions of the ball carrier through separately fitted components: the probability of passing or carrying, the risk of losing the ball in a one-second carry, the destination of a pass and its probability of completion. This page composes them into a single quantity, h, that answers one question: given the current state, how likely is the team to lose the ball during the carrier's next action? Because h is a probability, it has an absolute scale, which we divide into four bands. The bands are used in the [pressure case studies and player report](../07_case_studies/07_case_studies.ipynb).

## Possession as a discrete sequence of actions

PRESS segments a possession into a canonical set of on-ball actions, the segmentation described in [Possessions and checkpoints](../setup/possession_and_checkpoint_setup.ipynb). A player in possession either carries the ball for one second or passes it to a destination $j$, so the set of actions is

$$\mathcal A_0=\{\mathrm{carry}_{1s},\ \mathrm{pass}(j)\},$$

where a pass runs until its completion. The one-second carry unit is inspired by action-based EPV models [@fernandez2019; @fernandez2021; @stalpaert2022]. Shots are outside the action set.

Along a possession these actions form a sequence in which the team keeps the ball until, at some action, it loses it. A completed pass does not end the sequence: the ball moves to a teammate and the sequence continues across successive carriers. Let $T$ be the number of canonical actions until team possession is lost, and $s_n$ the game state at the $n$-th decision. The probability that the loss occurs at the next action, given that possession has been retained so far, is the discrete hazard

$$h(s_n)=P(T=n+1\mid T>n,\ s_n).$$

This is the quantity that answers the question: how fragile is the possession here? A carrier with free space and safe passing options has a small h, while a carrier whose carry is contested and whose passes are likely to fail has a large one. Defensive pressure acts on both branches at once, since it raises the risk of the carry and of the passes and it changes which action the player is likely to choose, and h summarizes these effects in a single probability. It is a **probability per decision**, indexed by actions and not by frames or seconds, and its scale depends on the one-second horizon of the carry.

PRESS builds h from its components and marginalizes over the average decision-making policy learned from observed play. It does not condition on the action the player actually chose, so h describes the difficulty of the situation and not the quality of the decision. Local geometry and kinematics enter execution, and passing options and context enter choice. A high value indicates difficult modeled alternatives for an average player in the observed context and not inevitable failure.

h is evaluated at the conventional checkpoints of the segmentation, one for each canonical action, and these define the process and all the results of this page. Since the components depend on the current state, h can also be computed at intermediate tracking frames, which provides descriptive observations of how difficulty evolves within a possession without adding actions.


## Mathematical definition

Condition all quantities on the current state $s$. Four separately fitted PRESS components enter:

* $\pi(s)=P(A=\mathrm{pass}\mid s)$ is the probability of passing from the [action-selection model](../04_action_selection/04_action_selection.ipynb), and $1-\pi(s)$ the probability of carrying;
* $d(s)=P(\mathrm{loss\ within\ 1s}\mid A=\mathrm{carry},s)$ is the loss probability of the one-second carry, from the [ball-drive risk model](../01_ball_drive_risk/01_ball_drive_risk.ipynb);
* $q_j(s)=P(J=j\mid A=\mathrm{pass},s)$ is the probability of destination $j$, from the [pass-destination model](../03_transition_model/03_pass_selection.ipynb);
* $p_j(s)=P(\mathrm{completion}\mid J=j,A=\mathrm{pass},s)$ is the completion probability of a pass to $j$, from the [pass-probability model](../02_pass_probability/02_pass_probability.ipynb).

Averaging pass failure over the destinations gives

$$f(s)=\sum_j q_j(s)\,\big(1-p_j(s)\big)=1-\sum_j q_j(s)\,p_j(s),$$

and the law of total probability over the two action branches gives

$$\boxed{h(s)=\pi(s)\,f(s)+\big(1-\pi(s)\big)\,d(s).}$$

The weighted pass contribution is $\pi f$ and the weighted carry contribution is $(1-\pi)d$; they add exactly to h and differ from the branch risks f and d themselves. A state can therefore have a high pass risk and a low h when passing is unlikely, and vice versa. The sum $\sum_j q_jp_j$ is the expected pass success of the action-selection features.

The composition is a project derivation, and the terminology follows the conditional failure probability of discrete-time survival analysis [@singer2003; @suresh2022discrete].


## Difficulty bands

The survival interpretation gives h an intuitive scale. If the current hazard stayed constant over the following actions, the probability of retaining possession for $n$ more actions would be $(1-h)^n$. The **half-life** is the number of actions after which this probability falls to 50%,

$$n_{1/2}=\frac{\log(1/2)}{\log(1-h)},$$

so the higher the hazard, the shorter the half-life. It is a reference scale in decisions and not in seconds, and it is not a prediction of how long the team keeps possession, since difficulty changes throughout play.

Inspired by the classification of defensive pressure of SkillCorner [@skillcorner2026], we define the **difficulty bands** from this scale. The cutoffs are the values of h whose half-life is exactly 8, 4 and 2 actions,

$$b_n=1-2^{-1/n},\qquad n=8,4,2.$$

This is a project design choice, and the thresholds are not learned from outcomes. The bands are defined on h and not on xLoss: they rest on a constant-hazard half-life construction, which requires a per-action risk over a fixed window, whereas xLoss is the probability of reaching an absorbing state, the terminal loss of the individual possession, and accumulates over a variable horizon, so the same thresholds would not apply to it. The API keeps the labels below.

| Band | h interval | Constant-risk half-life |
|---|---|---|
| None | $0\le h<0.082996$ | More than 8 decisions; infinite at h=0 |
| Low | $0.082996\le h<0.159104$ | $4<n_{1/2}\le8$ decisions |
| Medium | $0.159104\le h<0.292893$ | $2<n_{1/2}\le4$ decisions |
| High | $0.292893\le h\le1$ | At most 2 decisions |

The code uses the exact cutoffs, and a value on a cutoff enters the higher band. The same construction at n=1 gives a cutoff at h=0.5, which would define a fifth band. Only 36 of the 14,919 held-out states (0.24%) have h≥0.5, too few to evaluate a separate band reliably, so High is kept as the highest band. **None denotes the lowest modeled risk band.**

## Example

Auckland FC–Newcastle: L. Gillion loses possession during the carry following checkpoint 32937, and K. Grozos subsequently takes control.

The package computes h from the fitted components at a given state.


In [5]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
from PRESS import PRESS

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
match_dir = opendata / "data/processed/2026-10-15_training_v5_outfield/matches/1886347"
press = PRESS.from_skillcorner(
    match_dir, actions_path=match_dir / "1886347_action_selection_base.parquet"
)
frame_id = 32937

from PRESS.model import PassSelectionModel
from PRESS.extras import BCOS

xT_grid = PassSelectionModel.pretrained().xt_grid
area = float(BCOS(press.tracking_data).compute_bcos(frame_id)["BCOS area"])
result = press.compute_frame(
    frame_id, xT_grid, bcos_value=area,
    match_clock_min=53.56166666666666, score_difference_prefix=0.0,
)

from PRESS.visualization import plot_frame
from PRESS.utils.possession import hazard_to_band

h = result["HAZARD"]["Hazard"]
plot_frame(press, frame_id, title=f"Next-action hazard · frame {frame_id} · h={h:.1%} · {hazard_to_band(h)}")
plt.savefig("figures/vignette_06_h_real_pitch.png", dpi=170)
plt.savefig("figures/vignette_06_h_real_pitch.svg")
plt.close()


![h at a real canonical checkpoint.](../../figures/vignette_06_h_real_pitch.png)

## Validation of the bands

The table pools **14,919 canonical held-out action states** from the five outer test pairs. The observed target is the **failure of the current canonical action**, not terminal individual loss.

The band validation retains verified outfield, non-shot individual possessions whose terminal action belongs to the canonical action set. A pass supplies a checkpoint at its release, and a possession ending in a failed carry supplies the checkpoint at the start of the final failed action. Possessions ending in a foul suffered or a clearance have no terminal canonical action and are not part of this table, which leaves 14,919 states from 7,265 possessions. The pooled metrics of h and xLoss in the [composed-risk benchmark](../../benchmarks/06_composed_risk.qmd) use all 15,085 verified-outcome checkpoints, including the earlier carry checkpoints of these possessions.

Along the canonical sequence, the corresponding model-implied survival profile is $\widehat S_n=\prod_{k=0}^{n-1}[1-h(s_k)]$. Intermediate tracking snapshots do not add actions or survival factors, and h can still be computed from any usable current state.

| Band | Actions | Mean predicted h (95% interval) | Observed failures (95% interval) |
|---|---:|---:|---:|
| None | 8,659 | 2.93% [2.77%, 3.09%] | 3.40% [2.83%, 3.98%] |
| Low | 2,936 | 11.87% [11.82%, 11.93%] | 12.26% [10.72%, 14.09%] |
| Medium | 2,483 | 21.40% [21.27%, 21.50%] | 21.67% [19.20%, 24.41%] |
| High | 841 | 36.53% [35.88%, 37.21%] | 32.46% [29.19%, 36.27%] |

Observed failure increases monotonically across bands. In None, Low and Medium, mean predicted h lies within the bootstrap interval of the observed failure frequency. In High, mean predicted h is somewhat higher than observed (36.53% versus 32.46%), so the band is slightly pessimistic. The **paired predicted-minus-observed gap** is **4.07 percentage points**, with a 95% bootstrap interval of **−0.07 to 7.82 points**, which includes zero. The overall calibration of h in this band remains an open point, together with the noisiness of an estimate based on relatively few actions.

![Observed action-level failure and mean predicted h for the fixed bands.](../../figures/h_band_validation.png)

Error bars are 95% bootstrap intervals that resample whole matches [@efron1993bootstrap], because actions within a match are dependent. They describe between-match variability in this sample, conditional on fixed predictions, and with ten matches the 95% level is indicative. The [composed-risk benchmark](../../benchmarks/06_composed_risk.qmd) gives the bootstrap procedure and reports pooled metrics and calibration on the respective h and xLoss cohorts. No post-hoc calibrator is fitted on h.

## Extension to xLoss

The same Auckland FC–Newcastle possession ends with L. Gillion losing the ball and K. Grozos starting the next individual possession.

h concerns the next action. A complementary question concerns the eventual outcome of the current ball carrier's individual possession, which can extend beyond the next action. The distinction arises from the carry: a successful one-second carry preserves the individual possession but does not determine its outcome, since the player may keep carrying, pass or lose the ball later, while a successful pass ends the individual possession without a loss of team possession.

The [residual-risk model](../05_residual_risk/05_residual_risk.ipynb) accounts for this longer horizon with the residual risk

$$c(s)=P(\text{eventual individual possession loss}\mid s,\ \text{successful carry}).$$

Along the carry branch, the probability of an eventual loss combines the loss during the carry with the residual risk after its success, $d(s)+(1-d(s))\,c(s)$. Marginalizing over the same average decision-making policy as h gives the expected loss

$$\boxed{\mathrm{xLoss}(s)=\big(1-\pi(s)\big)\big[d(s)+\big(1-d(s)\big)c(s)\big]+\pi(s)\,f(s),}$$

or equivalently

$$\mathrm{xLoss}(s)=h(s)+\big(1-\pi(s)\big)\big(1-d(s)\big)\,c(s).$$

The additional term is the probability of a successful carry followed by an eventual loss of the individual possession. xLoss incorporates this longer horizon through $c(s)$, without modeling the subsequent sequence of decisions.

The two quantities are complementary. h characterizes the immediate difficulty of the situation and supports the difficulty bands, and xLoss estimates the probability that the current individual possession ends in a loss. Both are evaluated under the average decision-making policy, independently of the action actually performed.


In [9]:
from PRESS.model import ActionSelectionModel, ResidualRiskModel

state = press.get_state(frame_id)
context = press.get_frame_context(frame_id)
field = result["PPCF"]
action_model = ActionSelectionModel.pretrained()
residual_model = ResidualRiskModel.pretrained()
features = residual_model.build_features(
    action_model=action_model, time_since_start=1.0,
    x_bc=state["x_bc"], v_bc=state["v_bc"],
    r_atts=state["r_atts"], r_defs=state["r_defs"],
    attacking_side=state["attacking_side"], targets=field["grid"],
    transition_probabilities=field["Transition probabilities"],
    pass_success_probability=field["Pass probability"],
    bcos=area, ball_drive_loss=result["PI"]["Ball drive loss probability"],
    start_type_raw=context["start_type"], previous_action_raw=context["previous_action"],
    match_clock_min=53.56166666666666, score_difference_prefix=0.0,
    completed_canonical_drives=context["completed_canonical_drives"],
    pitch_length=state["pitch_length"], pitch_width=state["pitch_width"],
)
c = float(residual_model.predict_features(features)[0])

pi = result["ACTION_SELECTION"]["Pass probability"]
d = result["PI"]["Ball drive loss probability"]
xloss = h + (1 - pi) * (1 - d) * c
plot_frame(press, frame_id, title=f"Expected individual loss · frame {frame_id} · xLoss={xloss:.1%}")
plt.savefig("figures/vignette_06_xloss_real_pitch.png", dpi=170)
plt.savefig("figures/vignette_06_xloss_real_pitch.svg")
plt.close()


![xLoss at the same real checkpoint.](../../figures/vignette_06_xloss_real_pitch.png)

In [11]:
from PRESS.visualization import plot_frame

next_frame = 32947
plot_frame(press, next_frame, title="Next possession: K. Grozos · frame 32947")
plt.savefig("figures/vignette_06_xloss_next_pitch.png", dpi=170)
plt.savefig("figures/vignette_06_xloss_next_pitch.svg")
plt.close()


![Beginning of the next individual possession after the observed loss.](../../figures/vignette_06_xloss_next_pitch.png)

The metrics, calibration and conventions of h and xLoss are reported in the [composed-risk benchmark](../../benchmarks/06_composed_risk.qmd).
